In [ ]:
module chebyshev_method

    using LinearAlgebra

    export chebyshev_D
    # Chebyshev compute D = differentiation matrix, x = Chebyshev grid

    function  chebyshev_D(N)

        if N==0
            D = 0;
            x = 1;
            return D, x
        else
            θ = range(0,length=N+1,stop=pi)
            x = reshape(-cos.(θ), N+1, 1)
            c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
            X = repeat(x, 1, N+1);
            dX = X - X';
            D = (c * (1 ./ c)') ./ (dX .+ I(N+1));   # off-diagonal entries
            D = D - diagm(vec(sum(D, dims=2)));      # diagonal entries
            return D, x
        end
    end
end

In [ ]:
N = 99;      # Number of interior points
R = 285.36;    # Reynolds number
ω  = 0;     # Input frequency
β  = 0.07759

In [ ]:
using .chebyshev_method
using LinearAlgebra
using NonlinearEigenproblems
using Plots
using ToeplitzMatrices
    D,x=chebyshev_D(N)
    # for i = 1:N+1
    #     x[i,1]=200*x[i,1]+200
    # end
    # D=0.005*D
    for i=1:N+1
        D[i,:]=D[i,:].*((2*x[i]^3-x[i]^2+3*x[i]-4)^2/(20*(6*x[i]^2-2*x[i]+3)))
    end
    for i=1:N+1
        x[i]=(4*x[i]^3-2*x[i]^2+6*x[i]+12)/(-2*x[i]^3+x[i]^2-3*x[i]+4)
        if x[i]>10
            x[i]=10
        end
    end
    D2=D^2;
    D4=D^4;

In [ ]:
#interpolate data
using LinearAlgebra
using BSplineKit
    f = open( "Ekman.txt", "r" )
    n = countlines( f )
    seekstart( f )
    data = zeros(n,6)
    for i = 1:n
        z,w,u,v,du,dv = split( readline( f ), " " )  # 读取每一行数据并用split函数将数据“剥离”开来
        data[i,1] = parse(Float64,z)
        data[i,2] = parse(Float64,w)
        data[i,3] = parse(Float64,u)
        data[i,4] = parse(Float64,v)  # 将字符串转化为数字
        data[i,5] = parse(Float64,du)
        data[i,6] = parse(Float64,dv)
    end

    close( f )

    t=data[:,1]
    w=data[:,2]
    u=data[:,3]
    v=data[:,4]
    du=data[:,5]
    dv=data[:,6]
    itpw=itp = BSplineKit.interpolate(t, w , BSplineOrder(4))
    itpu=itp = BSplineKit.interpolate(t, u , BSplineOrder(4))
    itpv=itp = BSplineKit.interpolate(t, v , BSplineOrder(4))
    itpdu=itp = BSplineKit.interpolate(t, du , BSplineOrder(4))
    itpdv=itp = BSplineKit.interpolate(t, dv , BSplineOrder(4));

In [ ]:
    U=zeros(N+1,1)
    V=zeros(N+1,1)
    W=zeros(N+1,1)
    dU=zeros(N+1,1)
    dV=zeros(N+1,1)
    dW=zeros(N+1,1)
    for i=1:N+1
        U[i,1]=itpu(x[i])
        V[i,1]=itpv(x[i])
        W[i,1]=itpw(x[i])
        dU[i,1]=itpdu(x[i])
        dV[i,1]=itpdv(x[i])
    end
    dW=-2*U;
    ddU=D2*U;
    ddV=D2*V;
    

In [ ]:
using .chebyshev_method
using LinearAlgebra
using NonlinearEigenproblems
using SparseArrays
# D,x=chebyshev_D(N)
# D2=D^2
# U=1 .-x.^2
# dU=-2 .*x
# ddU=-2*ones(N+1)
# W=dW=zeros(N+1)
# V=dV=ddV=zeros(N+1)
A=D2-(β^2)*I(N+1)
L0_1= im*A^2 + R*β*V.*A - R*ω.*A - R*β*ddV.*I(N+1) + im*ddU.*I(N+1) - Ro*im*W.*A*D - Ro*im*dW.*A - Ro*im*U.*D2
L0_2=(2Ro*V.+Co).*D + 2*Ro*dV.*I(N+1)
L0_3=(2Ro*V.+Co).*D+ im*R*β*dU.*I(N+1)
L0_4= im*A + R*β*V.*I(N+1) - R*ω*I(N+1) - im*Ro*W.*D - im*Ro*U.*I(N+1)
L1_1= -(1/R).*A + R*U.*A + im*β*V.*I(N+1) - im*ω*I(N+1) - R*ddU.*I(N+1) + Ro*(1/R)*W.*D + Ro*(1/R)*dW.*I(N+1)
L1_2= zeros(N+1,N+1)
L1_3= -1*im*R*dV.*I(N+1)
L1_4= R*U.*I(N+1)
L2_1= -2*im*A + ω*R*I(N+1) - β*R*V.*I(N+1) + im*U.*I(N+1) + Ro*im*W.*D + Ro*im*dW.*I(N+1)
L2_2= L1_2
L2_3= L1_2
L2_4= -im*I(N+1)
L3_1= (1/R)*I(N+1) - R*U.*I(N+1)
L3_2= L3_3=L3_4=L1_2
L4_1= im*I(N+1)
L4_2= L4_3=L4_4=L1_2
L2_4= Matrix{ComplexF64}(L2_4)
# L0_1=L0_1[2:N,2:N];L0_2=L0_2[2:N,2:N];L0_3=L0_3[2:N,2:N];L0_4=L0_4[2:N,2:N];
# L1_1=L1_1[2:N,2:N];L1_2=L1_2[2:N,2:N];L1_3=L1_3[2:N,2:N];L1_4=L1_4[2:N,2:N];
# L2_1=L2_1[2:N,2:N];L2_2=L2_2[2:N,2:N];L2_3=L2_3[2:N,2:N];L2_4=L2_4[2:N,2:N];
# L3_1=L3_1[2:N,2:N];L3_2=L3_2[2:N,2:N];L3_3=L3_3[2:N,2:N];L3_4=L3_4[2:N,2:N];
# L4_1=L4_1[2:N,2:N];L4_2=L4_2[2:N,2:N];L4_3=L4_3[2:N,2:N];L4_4=L4_4[2:N,2:N];
A0=[L0_1 L0_2 ;L0_3 L0_4 ;]
A1=[L1_1 L1_2 ;L1_3 L1_4 ;]
A2=[L2_1 L2_2 ;L2_3 L2_4 ;]
A3=[L3_1 L3_2 ;L3_3 L3_4 ;]
A4=[L4_1 L4_2 ;L4_3 L4_4 ;]
A0=A0[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A1=A1[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A2=A2[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A3=A3[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
A4=A4[setdiff(1:end , (1,2,N,N+1,N+2,2N+2)),setdiff(1:end , (1,2,N,N+1,N+2,2N+2))]
# A0=A0[setdiff(1:end , (1,N+1,N+2,2N+2)),setdiff(1:end , (1,N+1,N+2,2N+2))]
# A1=A1[setdiff(1:end , (1,N+1,N+2,2N+2)),setdiff(1:end , (1,N+1,N+2,2N+2))]
# A2=A2[setdiff(1:end , (1,N+1,N+2,2N+2)),setdiff(1:end , (1,N+1,N+2,2N+2))]
# A3=A3[setdiff(1:end , (1,N+1,N+2,2N+2)),setdiff(1:end , (1,N+1,N+2,2N+2))]
# A4=A4[setdiff(1:end , (1,N+1,N+2,2N+2)),setdiff(1:end , (1,N+1,N+2,2N+2))]
A0=sparse(A0);
A1=sparse(A1);
A2=sparse(A2);
A3=sparse(A3);
A4=sparse(A4);

In [ ]:
nep = PEP([A0,A1,A2,A3,A4]); #Create a PEP object

In [ ]:
norm.(get_Av(nep))

In [ ]:
sc=10
nep1 = shift_and_scale(nep,scale=sc);
mult_scale = norm(nep1.A[end]);
nep2 = PEP(nep1.A ./ mult_scale);
norm.(get_Av(nep2))

In [2]:
λ1,v1 = iar(nep2,neigs=20,maxit=500);
λtotal = [λ1];

UndefVarError: UndefVarError: `iar` not defined

In [ ]:
λ_orig = sc*λtotal

In [ ]:
plot(real.(λ_orig),imag.(λ_orig),seriestype=:scatter,xlabel="Cr",ylabel="Ci")